# Experiment 4: YOLO Object Detection

This notebook implements YOLO object detection based on the practical.
It includes:
- image inference (default, notebook-friendly)
- optional webcam inference loop (for local GUI environments)
        

In [ ]:
!pip install numpy matplotlib "ultralytics>=8.0" "opencv-python-headless>=4.8"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 703.9 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 3.5 MB/s eta 0:00:00
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 7.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 3.6 MB/s eta 0:00:0000:0100:01m
Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.2/355.2 kB 5.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 5.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from urllib.request import urlretrieve
from ultralytics import YOLO


In [ ]:
# Load pretrained YOLO model.
model = YOLO("yolov8n.pt")

# Use local image.jpg if present; otherwise download a sample image.
image_path = Path("image.jpg")
if not image_path.exists():
    sample_url = "https://ultralytics.com/images/bus.jpg"
    urlretrieve(sample_url, image_path)
    print(f"Downloaded sample image to: {image_path.resolve()}")

img = cv2.imread(str(image_path))
if img is None:
    raise FileNotFoundError(f"Could not read image: {image_path}")

results = model(img)
annotated = results[0].plot()

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLO Detection Result")
plt.show()
        

In [ ]:
# Print detected objects and confidence.
for box in results[0].boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    print(f"Object: {model.names[cls_id]} | Confidence: {conf:.2f}")
        

In [ ]:
# Optional: real-time webcam detection.
# Run this block only if your environment has webcam access and GUI display.
RUN_WEBCAM_DEMO = False

if RUN_WEBCAM_DEMO:
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam (index 0)")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        webcam_results = model(frame)
        annotated_frame = webcam_results[0].plot()
        cv2.imshow("YOLO Object Detection", annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
        